# ⏱️ Forecasting com TimesFM

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | TimesFM — Foundation Model decoder-only para séries temporais (zero-shot) |
| **Biblioteca** | `timesfm` 1.3.0 — `TimesFm` |
| **Versão do modelo** | `google/timesfm-1.0-200m-pytorch` (200M parâmetros) |
| **Hiperparâmetros configurados** | `FREQUENCY=1` (baixa/mensal), `horizon_len=128` (interno, saída cortada para 3 meses), `per_core_batch_size=32`, `backend='cpu'`, `SEED=42` |
| **Busca de hiperparâmetros** | Não (zero-shot — sem fine-tuning) |
| **Critério de seleção** | N/A |
| **Séries utilizadas** | 29 séries com treino ≥ 34 observações (`len(train_series) < TEST_SIZE + 10`) |
| **Horizonte** | 3 meses (`HORIZON = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses |
| **Reprodutibilidade** | `SEED = 42` (`random.seed` + `np.random.seed` + `torch.manual_seed` + `cuda.manual_seed_all`) |
| **Referência** | Das, A. et al. (2024). A Decoder-only Foundation Model for Time-series Forecasting. *arXiv:2310.10688*. Google Research. |

---

In [ ]:
# ── Semente global para reprodutibilidade ──
import random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"🔒 Seed fixada: {SEED} (random + numpy + torch)")

## 1. Instalação de Dependências

In [1]:
# Instalação do TimesFM
# Requer Python 3.10+ e PyTorch/JAX
%pip install timesfm -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importação de Bibliotecas

In [2]:
import pandas as pd
import numpy as np
import timesfm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

print(f"TimesFM importado com sucesso!")

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


C:\Users\phill\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded PyTorch TimesFM, likely because python version is 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
TimesFM importado com sucesso!


## 3. Carregamento dos Dados

In [3]:
# Carregar base de dados
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)

print(f"📊 Shape: {df.shape}")
print(f"📅 Período: {df.index.min().strftime('%Y-%m')} a {df.index.max().strftime('%Y-%m')}")
print(f"\n📈 Séries disponíveis:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2}. {col}")

📊 Shape: (108, 35)
📅 Período: 2017-01 a 2025-12

📈 Séries disponíveis:
    1. IBC_Br
    2. Selic
    3. Cambio_USDBRL
    4. Desemprego
    5. Brent_USD
    6. Soja_USD
    7. Minerio_USD
    8. Ibovespa
    9. ICC_FGV
   10. Credito_Total
   11. Inadimplencia
   12. Massa_Salarial
   13. CPI_USA
   14. Prod_Ind_USA
   15. Cafe_USD
   16. Ouro_USD
   17. GasNatural_USD
   18. Cobre_USD
   19. ETF_Emergentes
   20. IGP_DI
   21. INCC
   22. ICE_Empresarial
   23. Housing_Starts_EUA
   24. Dollar_Index_Fed
   25. PMS_Volume
   26. PMC_Ampliado
   27. IGPM
   28. INPC
   29. M2
   30. Divida_PIB
   31. Vendas_Varejo
   32. Balanca_Comercial
   33. NUCI_FGV
   34. EAI_Emprego_Ind
   35. SP500


In [4]:
# Inicializar TimesFM
# Modelo: timesfm-1.0-200m-pytorch (versão PyTorch)

print("⏳ Carregando modelo TimesFM-1.0-200M (PyTorch)...")

# Configurar e carregar o modelo - usando versão PyTorch
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="cpu",
        per_core_batch_size=32,
        horizon_len=128,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id="google/timesfm-1.0-200m-pytorch"
    ),
)

print("✅ Modelo carregado!")

⏳ Carregando modelo TimesFM-1.0-200M (PyTorch)...


Fetching 3 files: 100%|██████████| 3/3 [00:00<?, ?it/s]


✅ Modelo carregado!


## 4. Configuração do Experimento

In [5]:
# Configurações
HORIZON = 3              # Prever 3 meses
TEST_SIZE = 24           # Últimos 24 meses para teste
FREQUENCY = 1            # Frequência: 0=high freq (second/minute), 1=medium (hour/day), 2=low (week/month)

print("📐 Configuração:")
print(f"   • Horizonte: {HORIZON} meses")
print(f"   • Teste: {TEST_SIZE} meses")

print(f"   • Frequência: {FREQUENCY} (0=alta, 1=média, 2=baixa/mensal)")
print(f"   • Modelo: TimesFM-1.0-200M")

📐 Configuração:
   • Horizonte: 3 meses
   • Teste: 12 meses
   • Frequência: 1 (0=alta, 1=média, 2=baixa/mensal)
   • Modelo: TimesFM-1.0-200M


In [6]:
def calculate_metrics(y_true, y_pred):
    """
    Calcula métricas de avaliação.
    """
    mse = np.mean((y_true - y_pred)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}


def forecast_with_timesfm(model, series, horizon, freq=1):
    """
    Faz previsão usando TimesFM.
    
    Args:
        model: modelo TimesFM
        series: array com os valores históricos
        horizon: número de passos à frente
        freq: frequência (0=alta, 1=média, 2=baixa)
        
    Returns:
        point_forecast: previsão pontual
        quantile_forecast: previsões quantílicas (se disponível)
    """
    # TimesFM espera lista de arrays
    forecast_input = [series.astype(np.float32)]
    frequency_input = [freq]
    
    # Fazer previsão (horizon controlado via horizon_len no hparams, fatiamos depois)
    point_forecast, quantile_forecast = model.forecast(
        forecast_input,
        freq=frequency_input,
    )
    
    return point_forecast[0][:horizon], quantile_forecast


print("✅ Funções auxiliares definidas!")

✅ Funções auxiliares definidas!


In [7]:
print("="*70)
print("⏱️ EXECUTANDO TIMESFM PARA TODAS AS SÉRIES")
print("="*70)

# Excluir PIM (70% missing) e IPCA_mensal (variável-alvo) — padrão de todos os modelos
ALL_SERIES = [s for s in df.columns if s not in ['PIM', 'IPCA_mensal']]
all_results = []

# Split global treino/teste (padrão de todos os notebooks da dissertação)
train_df = df.iloc[:-TEST_SIZE]
test_df = df.iloc[-TEST_SIZE:]

print(f"   Séries selecionadas: {len(ALL_SERIES)} (excluídas PIM e IPCA_mensal)")
print(f"   Train: {train_df.index[0].strftime('%Y-%m')} a {train_df.index[-1].strftime('%Y-%m')} ({len(train_df)} obs)")
print(f"   Test:  {test_df.index[0].strftime('%Y-%m')} a {test_df.index[-1].strftime('%Y-%m')} ({len(test_df)} obs)")

for idx, series_name in enumerate(ALL_SERIES, 1):
    print(f"\n[{idx}/{len(ALL_SERIES)}] 📈 {series_name}")
    print("-"*50)
    
    try:
        # Extrair treino e teste após split global, depois dropna por série
        train_series = train_df[series_name].dropna()
        test_series = test_df[series_name].dropna()
        
        if len(train_series) < TEST_SIZE + 10:
            print(f"   ⚠️ Série muito curta: {len(train_series)} observações treino")
            continue
        
        if len(test_series) == 0:
            print(f"   ⚠️ Sem dados de teste")
            continue
        
        # Walk-forward validation no período de teste
        all_preds = []
        all_actuals = []
        all_dates = []
        
        for i in range(0, len(test_series), HORIZON):
            # Contexto: treino + dados de teste já observados
            if i > 0:
                context = np.concatenate([train_series.values, test_series.values[:i]])
            else:
                context = train_series.values
            
            # Número de passos a prever
            n_steps = min(HORIZON, len(test_series) - i)
            
            # Fazer previsão
            pred, _ = forecast_with_timesfm(
                tfm,
                context, 
                horizon=n_steps, 
                freq=FREQUENCY
            )
            
            # Valores reais
            actuals = test_series.values[i:i+n_steps]
            
            all_preds.extend(pred[:n_steps])
            all_actuals.extend(actuals)
            all_dates.extend(test_series.index[i:i+n_steps])
        
        all_preds = np.array(all_preds)
        all_actuals = np.array(all_actuals)
        
        # Calcular métricas
        metrics = calculate_metrics(all_actuals, all_preds)
        
        print(f"   ✅ MAE: {metrics['MAE']:.4f} | RMSE: {metrics['RMSE']:.4f} | MAPE: {metrics['MAPE']:.2f}%")
        
        # Guardar resultados
        all_results.append({
            'Serie': series_name,
            'N_Obs': len(train_series) + len(test_series),
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
            'N_Pontos': len(all_preds),
            'preds': all_preds,
            'actuals': all_actuals,
            'dates': all_dates
        })
        
    except Exception as e:
        print(f"   ❌ Erro: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*70)
print(f"✅ EXECUÇÃO COMPLETA: {len(all_results)}/{len(ALL_SERIES)} séries")
print("="*70)

⏱️ EXECUTANDO TIMESFM PARA TODAS AS SÉRIES
   Séries selecionadas: 35 (excluídas PIM e IPCA_mensal)
   Train: 2017-01 a 2024-12 (96 obs)
   Test:  2025-01 a 2025-12 (12 obs)

[1/35] 📈 IBC_Br
--------------------------------------------------
   ✅ MAE: 1.1519 | RMSE: 1.4197 | MAPE: 1.06%

[2/35] 📈 Selic
--------------------------------------------------
   ✅ MAE: 0.8233 | RMSE: 0.9148 | MAPE: 5.68%

[3/35] 📈 Cambio_USDBRL
--------------------------------------------------
   ✅ MAE: 0.1297 | RMSE: 0.1553 | MAPE: 2.30%

[4/35] 📈 Desemprego
--------------------------------------------------
   ✅ MAE: 0.5243 | RMSE: 0.7300 | MAPE: 8.83%

[5/35] 📈 Brent_USD
--------------------------------------------------
   ✅ MAE: 4.9399 | RMSE: 6.6748 | MAPE: 7.44%

[6/35] 📈 Soja_USD
--------------------------------------------------
   ✅ MAE: 14.8095 | RMSE: 20.6325 | MAPE: 3.79%

[7/35] 📈 Minerio_USD
--------------------------------------------------
   ✅ MAE: 3.7300 | RMSE: 4.3890 | MAPE: 3.61%

[8/35

## 5. Resultados e Métricas

In [8]:
# Criar DataFrame com resultados
df_results = pd.DataFrame([{
    'Serie': r['Serie'],
    'N_Obs': r['N_Obs'],
    'MAE': r['MAE'],
    'RMSE': r['RMSE'],
    'MAPE': r['MAPE'],
    'N_Pontos': r['N_Pontos']
} for r in all_results])

print("="*70)
print("📊 RESULTADOS FINAIS - TIMESFM")
print("="*70)
print(f"\nModelo: TimesFM | Horizonte: {HORIZON} meses | Teste: {TEST_SIZE} meses")

print(f"\n📈 Estatísticas Gerais:")
print(f"   • Séries processadas: {len(df_results)}")
print(f"   • MAE médio:  {df_results['MAE'].mean():.4f}")
print(f"   • RMSE médio: {df_results['RMSE'].mean():.4f}")
print(f"   • MAPE médio: {df_results['MAPE'].mean():.2f}%")

print("\n📊 Resultados por Série:")
print(df_results[['Serie', 'MAE', 'RMSE', 'MAPE']].sort_values('MAPE').to_string(index=False))

📊 RESULTADOS FINAIS - TIMESFM

Modelo: TimesFM | Horizonte: 3 meses | Teste: 12 meses

📈 Estatísticas Gerais:
   • Séries processadas: 35
   • MAE médio:  8256468.2131
   • RMSE médio: 9146847.0300
   • MAPE médio: 12984947.93%

📊 Resultados por Série:
             Serie          MAE         RMSE         MAPE
      Prod_Ind_USA 4.112664e-01 5.367257e-01 4.060279e-01
           CPI_USA 2.577100e+00 2.888771e+00 8.003359e-01
    Massa_Salarial 3.198479e+03 3.413534e+03 8.993605e-01
            IBC_Br 1.151931e+00 1.419664e+00 1.060026e+00
     Vendas_Varejo 1.229551e+00 1.560344e+00 1.157345e+00
        Divida_PIB 7.749216e-01 8.561428e-01 1.225762e+00
        PMS_Volume 1.573431e+00 1.810486e+00 1.454014e+00
          NUCI_FGV 1.336894e+00 1.485792e+00 1.639015e+00
   ICE_Empresarial 1.841550e+00 2.096044e+00 1.762381e+00
     Credito_Total 6.904608e+04 8.751834e+04 1.803920e+00
                M2 2.882887e+08 3.193235e+08 2.051954e+00
  Dollar_Index_Fed 2.680443e+00 2.893425e+00 2.1989

In [9]:
# Ranking por MAPE
print("\n🏆 Ranking por MAPE (menor = melhor):")
print("="*50)

df_sorted = df_results.sort_values('MAPE')
for i, (_, row) in enumerate(df_sorted.iterrows(), 1):
    emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{emoji} {i:2}. {row['Serie']:20} : {row['MAPE']:.2f}%")


🏆 Ranking por MAPE (menor = melhor):
🥇  1. Prod_Ind_USA         : 0.41%
🥈  2. CPI_USA              : 0.80%
🥉  3. Massa_Salarial       : 0.90%
    4. IBC_Br               : 1.06%
    5. Vendas_Varejo        : 1.16%
    6. Divida_PIB           : 1.23%
    7. PMS_Volume           : 1.45%
    8. NUCI_FGV             : 1.64%
    9. ICE_Empresarial      : 1.76%
   10. Credito_Total        : 1.80%
   11. M2                   : 2.05%
   12. Dollar_Index_Fed     : 2.20%
   13. Cambio_USDBRL        : 2.30%
   14. SP500                : 2.87%
   15. Minerio_USD          : 3.61%
   16. Inadimplencia        : 3.66%
   17. Soja_USD             : 3.79%
   18. Housing_Starts_EUA   : 3.96%
   19. ICC_FGV              : 4.97%
   20. Selic                : 5.68%
   21. ETF_Emergentes       : 5.74%
   22. Ibovespa             : 6.63%
   23. Ouro_USD             : 7.09%
   24. Brent_USD            : 7.44%
   25. Cobre_USD            : 8.55%
   26. Desemprego           : 8.83%
   27. GasNatural_USD       :

## 6. Exportação de Resultados

In [10]:
# Adicionar classificação baseada no MAPE (padrão da dissertação)
def classificar(mape):
    if mape < 10:
        return '⭐ Excelente'
    elif mape < 20:
        return '✅ Muito Bom'
    elif mape < 30:
        return '👍 Bom'
    elif mape < 50:
        return '⚠️ Regular'
    else:
        return '❌ Difícil'

df_results['Classificacao'] = df_results['MAPE'].apply(classificar)

# Salvar resultados com nomes padronizados para compatibilidade com consolidação
csv_path = 'resultados_timesfm.csv'
df_results[['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']].to_csv(csv_path, index=False)
print(f"💾 Resultados salvos em: {csv_path}")
print(f"   Colunas: {['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']}")

# Salvar previsões individuais (Serie, Data, Previsao) — padrão dos demais notebooks
previsoes_rows = []
for r in all_results:
    for d, p in zip(r['dates'], r['preds']):
        previsoes_rows.append({'Serie': r['Serie'], 'Data': str(d)[:10], 'Previsao': p})
df_prev = pd.DataFrame(previsoes_rows)
df_prev.to_csv('previsoes_timesfm.csv', index=False)
print(f"💾 Previsões salvas em: previsoes_timesfm.csv ({len(df_prev)} linhas)")

💾 Resultados salvos em: resultados_timesfm.csv
   Colunas: ['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']
💾 Previsões salvas em: previsoes_timesfm.csv (420 linhas)


## 7. Visualização: Ranking MAPE por Série

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = df_results.sort_values('MAPE')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df['Serie'])
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'TimesFM — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('timesfm_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: timesfm_mape_por_serie.png")

## 8. Visualização: Real vs. Projetado (Top 6 Séries)

In [ ]:
# ── Gráfico: Real vs. Projetado (Top 6 Séries por MAPE) ──
top6 = sorted_df['Serie'].head(6).tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, sn in zip(axes.flatten(), top6):
    r = next(x for x in all_results if x['Serie'] == sn)

    # Valores reais (teste)
    ax.plot(r['actuals'], 'b-o',
            label='Real', markersize=4, linewidth=2)

    # Previsões do modelo
    ax.plot(r['preds'], 'r--s',
            label='Previsão', markersize=4, linewidth=2)

    ax.set_title(f"{sn}\nMAPE: {r['MAPE']:.1f}%", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

axes.flatten()[0].legend(fontsize=8)
fig.suptitle('TimesFM — Real vs. Projetado (6 Melhores Séries)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('timesfm_previsoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: timesfm_previsoes.png")

In [13]:
# Resumo final
print("\n" + "="*70)
print("📋 RESUMO FINAL - TIMESFM")
print("="*70)
print(f"\n🎯 MAPE Médio: {df_results['MAPE'].mean():.2f}%")
print(f"\n🏆 Melhor série:  {df_sorted.iloc[0]['Serie']} ({df_sorted.iloc[0]['MAPE']:.2f}%)")
print(f"📉 Pior série:    {df_sorted.iloc[-1]['Serie']} ({df_sorted.iloc[-1]['MAPE']:.2f}%)")
print(f"\n📊 Séries com MAPE < 10%: {len(df_results[df_results['MAPE'] < 10])}")
print(f"📊 Séries com MAPE < 20%: {len(df_results[df_results['MAPE'] < 20])}")
print(f"📊 Séries com MAPE > 50%: {len(df_results[df_results['MAPE'] > 50])}")


📋 RESUMO FINAL - TIMESFM

🎯 MAPE Médio: 12984947.93%

🏆 Melhor série:  Prod_Ind_USA (0.41%)
📉 Pior série:    INPC (454471901.95%)

📊 Séries com MAPE < 10%: 26
📊 Séries com MAPE < 20%: 29
📊 Séries com MAPE > 50%: 5
